# Graphs

**Geometry type:** `graph`

Arbitrary vertex–edge graphs embedded in 3-D space — vascular networks,
synaptic connectivity graphs, any structure with cycles.

1. Generate a synthetic vascular network
2. Write with node + edge attributes
3. Open lazily and inspect
4. Read all nodes and edges
5. Spatial bounding-box query
6. Validate

In [1]:
import numpy as np, tempfile, os
from zarr_vectors.lazy import open_zvr
from zarr_vectors.types.graphs import write_graph, read_graph
from zarr_vectors.validate import validate

_tmpdir = tempfile.mkdtemp(prefix="zvf_examples_")
STORE = os.path.join(_tmpdir, "vessels.zarrvectors")
print("Store:", STORE)

Store: /tmp/zvf_examples_oxa9kwl1/vessels.zarrvectors


## 1 · Generate a synthetic vascular network graph

In [2]:
rng = np.random.default_rng(2)
n_nodes = 3_000

# Nodes scattered in a 600³ µm volume
positions = rng.uniform(0, 600, (n_nodes, 3)).astype(np.float32)

# Edges: connect each node to its ~4 nearest neighbours (simplified).
# In practice, use a real graph from a segmentation pipeline.
from scipy.spatial import KDTree
tree = KDTree(positions)
_, nbrs = tree.query(positions, k=5)   # k=1 is self; skip it
src = np.repeat(np.arange(n_nodes, dtype=np.int32), 4)
dst = nbrs[:, 1:].flatten().astype(np.int32)

# Remove self-loops, dedupe, store as canonical [min, max] for undirected
mask  = src != dst
edges = np.column_stack([src[mask], dst[mask]])
edges = np.sort(edges, axis=1)
edges = np.unique(edges, axis=0).astype(np.int32)

# Per-node attributes
diameter = rng.uniform(1, 20, n_nodes).astype(np.float32)
flow     = rng.uniform(0, 1, n_nodes).astype(np.float32)

# Per-edge attribute: length (computed from positions)
edge_len = np.linalg.norm(positions[edges[:, 0]] - positions[edges[:, 1]], axis=1).astype(np.float32)

print(f"nodes : {n_nodes:,}")
print(f"edges : {len(edges):,}")
print(f"mean degree : {2 * len(edges) / n_nodes:.1f}")

nodes : 3,000
edges : 7,428
mean degree : 5.0


## 2 · Write the graph

In [3]:
write_graph(
    STORE,
    positions=positions,
    edges=edges,
    chunk_shape=(150.0, 150.0, 150.0),
    bin_shape=(37.5, 37.5, 37.5),
    is_tree=False,
    node_attributes={
        "diameter": diameter,
        "flow":     flow,
    },
    edge_attributes={
        "length":   edge_len,
    },
)
print("Write complete.")

Write complete.


## 3 · Open lazily and inspect

In [4]:
store = open_zvr(STORE)
print(f"geometry_types: {store.geometry_types}")
print(f"ndim          : {store.ndim}")
print(f"chunk_shape   : {store.chunk_shape}")
print(f"vertex_count  : {store[0].vertex_count:,}")
lo, hi = np.array(store.bounds[0]), np.array(store.bounds[1])
print(f"bounds        : lo={lo.round(1)}  hi={hi.round(1)}")

geometry_types: ['graph']
ndim          : 3
chunk_shape   : (150.0, 150.0, 150.0)
vertex_count  : 3,000
bounds        : lo=[0.  0.  0.8]  hi=[599.8 599.9 600. ]


## 4 · Read all nodes and edges

In [5]:
result = read_graph(STORE)
print(f"node_count : {result['node_count']:,}")
print(f"edge_count : {result['edge_count']:,}")
print(f"positions  : {result['positions'].shape}")
print(f"edges      : {result['edges'].shape}   (remapped to output indices)")

node_count : 3,000
edge_count : 7,428
positions  : (3000, 3)
edges      : (7428, 2)   (remapped to output indices)


## 5 · Spatial bounding-box query

In [6]:
lo = np.array([200.0, 200.0, 200.0])
hi = np.array([400.0, 400.0, 400.0])

bbox_result = read_graph(STORE, bbox=(lo, hi))
print(f"Nodes in 200³ µm bbox: {bbox_result['node_count']:,}")
print(f"Edges (both ends in bbox, remapped to bbox indices): {bbox_result['edge_count']:,}")

Nodes in 200³ µm bbox: 98
Edges (both ends in bbox, remapped to bbox indices): 173


## 6 · Validate

In [7]:
rv = validate(STORE, level=3)
print(rv.summary())


Level 3 validation: PASS
  25 passed, 0 warnings, 0 errors


## Summary

`graph` stores arbitrary node-edge graphs. The current implementation is
undirected (sort edges canonically before writing if you want consistent
ordering).

| Step | API |
|------|-----|
| Write | `write_graph(path, positions, edges, node_attributes, edge_attributes, is_tree=False)` |
| Read all | `read_graph(path)` → `{positions, edges, node_count, edge_count}` |
| Bbox query | `read_graph(path, bbox=(lo, hi))` |
| Read by ID | `read_graph(path, object_ids=[...])` |